# Clase 08 — Modelos de lenguaje n-grama

**Curso:** NLP Clásico con Python · **Bloque 3:** Estructura y modelos del lenguaje
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/08_modelos_lenguaje_ngramas.md`

## Qué vas a ver en esta clase

- Qué es un **modelo de lenguaje** (la probabilidad de una secuencia de palabras), la **regla de la cadena** y la **hipótesis de Markov**.
- Estimar **bigramas y trigramas por conteo**, primero a mano con un corpus de cuatro frases y luego con dos extractos literarios (español e italiano) y un modelo escrito por nosotros en Python puro.
- **Suavizado** (Laplace), interpolación y la idea de *backoff*, comprobando numéricamente que las probabilidades condicionadas suman 1.
- **Perplejidad** sobre datos de prueba, y por qué solo es comparable con el mismo vocabulario, la misma tokenización y el mismo tratamiento de las palabras fuera de vocabulario (OOV).
- **Generar** texto muestreando el modelo y ver sus límites.

**Requisitos:** clases 2 y 4 (tokenización, conteo de n-gramas). **Librerías:** `numpy` y `matplotlib`; el modelo no usa ninguna librería de modelos de lenguaje.
**Datos:** `datos/gutenberg/quijote_cap01-14.txt` y `datos/gutenberg/promessi_sposi_cap01-06.txt` (en Colab, sube la carpeta `datos/`). Son textos literarios de dominio público de hace dos siglos o más: **no representan el español ni el italiano de hoy**. Todas las cifras son un *ejemplo ilustrativo*.

---
## Celda 0 — Setup

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("matplotlib", "matplotlib")
print("Librerías listas.")

In [ ]:
import os
import re
import math
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

random.seed(42)                          # semilla fija
np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

def tabla(cabeceras, filas, ancho=14):
    """Imprime una tabla de texto alineada."""
    print("".join(str(c).ljust(ancho) for c in cabeceras))
    print("-" * ancho * len(cabeceras))
    for fila in filas:
        print("".join((f"{x:.1f}" if isinstance(x, float) else str(x)).ljust(ancho) for x in fila))

print("Configuración lista.")

---
## 1. Un modelo de lenguaje a mano

Un **modelo de lenguaje** asigna una probabilidad a una secuencia de palabras. La **regla de la cadena** la descompone exactamente:

$$P(w_1,\dots,w_m)=\prod_{i=1}^{m} P(w_i\mid w_1,\dots,w_{i-1})$$

Como no podemos contar historias enteras (casi ninguna se repite), la **hipótesis de Markov** las recorta: en un bigrama, $P(w_i\mid w_1,\dots,w_{i-1})\approx P(w_i\mid w_{i-1})$, y en un trigrama se usan las dos palabras anteriores. Se estima por conteo: $P(w\mid v)=c(v,w)/c(v)$.

Corpus minúsculo (4 frases). Calcula a mano antes de ejecutar: `P(gato | un)`, y la probabilidad de la frase `un gato come pescado`.

In [ ]:
# datos-es
# Corpus minúsculo (4 frases por idioma) para calcular a mano
CHICO = {
    "spa": [["un", "gato", "come", "pescado"], ["un", "gato", "duerme"],
            ["un", "perro", "come", "carne"], ["un", "perro", "duerme"]],
    "ita": [["il", "gatto", "mangia", "pesce"], ["il", "gatto", "dorme"],
            ["il", "cane", "mangia", "carne"], ["il", "cane", "dorme"]],
}

In [ ]:
# Tokens especiales del modelo
INI, FIN, UNK = "<s>", "</s>", "<unk>"     # inicio de frase, fin de frase, palabra fuera de vocabulario (OOV)
ENCAB = re.compile(r"^(Capítulo|CAPITOLO)\b")   # dato-es

def capitulos(ruta):
    """Lee un extracto y devuelve una lista de capítulos; cada capítulo es una lista de párrafos (sin el encabezado)."""
    caps = []
    for parrafo in open(ruta, encoding="utf-8").read().split("\n\n"):
        parrafo = parrafo.strip()
        if not parrafo:
            continue
        if ENCAB.match(parrafo):
            caps.append([])
        else:
            caps[-1].append(parrafo)
    return caps

def oraciones(parrafos, puntuacion=False):
    """Divide en frases (. ! ? …) y en tokens en minúscula; por defecto los signos de puntuación no son tokens."""
    patron = r"\w+'|\w+|[,;:.!?¿¡«»—]" if puntuacion else r"\w+'|\w+"   # dato-es
    salida = []
    for p in parrafos:
        for o in re.split(r"(?<=[.!?…])\s+", p):
            toks = re.findall(patron, o.lower())
            if toks:
                salida.append(toks)
    return salida

def contar(oras, n):
    """Cuenta n-gramas con n-1 marcas <s> al inicio y una </s> al final. Devuelve (n-gramas, contextos)."""
    c, ctx = Counter(), Counter()
    for o in oras:
        seq = [INI] * (n - 1) + o + [FIN]
        for i in range(n - 1, len(seq)):
            c[tuple(seq[i - n + 1:i + 1])] += 1
            ctx[tuple(seq[i - n + 1:i])] += 1
    return c, ctx

def vocabulario(oras, minimo):
    """Palabras con al menos `minimo` apariciones en el entrenamiento."""
    cuenta = Counter(t for o in oras for t in o)
    return {t for t, k in cuenta.items() if k >= minimo}

def aplicar_vocab(oras, vocab):
    """Sustituye por <unk> lo que no está en el vocabulario."""
    return [[t if t in vocab else UNK for t in o] for o in oras]

In [ ]:
# Cálculo a mano con el corpus minúsculo (español; el italiano es idéntico salvo las palabras)
idioma = "spa"
oras = CHICO[idioma]
bi, ctx1 = contar(oras, 2)
print("Bigramas contados:")
for par, k in sorted(bi.items()):
    print("  ", par, k, "/", ctx1[par[:1]], "=", k / ctx1[par[:1]])

def p_bigrama(w, v):
    return bi[(v, w)] / ctx1[(v,)]

frase = oras[0] + [FIN]
p = p_bigrama(frase[0], INI)
for v, w in zip(frase, frase[1:]):
    p *= p_bigrama(w, v)
print("\nP(gato | un) =", p_bigrama("gato", "un"))
print("P(", " ".join(oras[0]), ") =", p)
assert p_bigrama("gato", "un") == 0.5 and p == 0.125             # coincide con el cálculo a mano: 1 · 1/2 · 1/2 · 1/2 · 1

### ¿Qué observas?

`P(gato | un) = 2/4`: `un` aparece 4 veces al comienzo de una frase (`<s> un`) y 2 de ellas van seguidas de `gato`. La frase completa vale `1 · 0,5 · 0,5 · 0,5 · 1 = 0,125`; el último factor `P(</s> | pescado)` es 1 porque `pescado` siempre cierra la frase. Sin la marca `<s>` no tendríamos la probabilidad de **empezar** con `un`, y sin `</s>` un modelo no sabría **cuándo terminar**: por eso las añadimos. (`<s>` e `</s>` son tokens especiales: marcan inicio y fin de frase; el tercero, `<unk>`, lo introducimos en la sección 2.)

---
## 2. Datos reales, tokens especiales y OOV

Definimos lo que usará todo el modelo (y que **debe ser idéntico** para entrenar y para evaluar):

- **Tokenización:** minúsculas; un token es una palabra (`\w+`), y en italiano un artículo o preposición elidido con apóstrofo (`dell'`, `l'`) es un token aparte. Los signos de puntuación **no** son tokens (más adelante veremos cómo cambia la perplejidad si lo fueran). Las frases se separan tras `.`, `!`, `?` o `…`.
- `<s>`: marca de inicio de frase (se antepone n−1 veces); `</s>`: fin de frase, que el modelo **predice**; `<unk>`: sustituye cualquier palabra **fuera de vocabulario (OOV)**.
- **Tratamiento del OOV:** el vocabulario son las palabras con al menos 2 apariciones en el **entrenamiento**; el resto (en entrenamiento y en prueba) pasa a `<unk>`. El vocabulario se fija con el entrenamiento y **no se toca** con los datos de prueba.
- **División por capítulos contiguos, no al azar:** entrenamiento = primeros capítulos, desarrollo = los siguientes (para elegir hiperparámetros) y prueba = los últimos. Es lo que ocurre en la práctica: el modelo se usa sobre texto que llega **después**.

**Advertencia:** un corpus literario de hace dos siglos no es el español ni el italiano de hoy (`lantejas`, `desta`, `mesmo`, `talchè`): lo que aprenda el modelo es la lengua de ese libro.

In [ ]:
CORPORA = {"spa": "gutenberg/quijote_cap01-14.txt", "ita": "gutenberg/promessi_sposi_cap01-06.txt"}
# Capítulos contiguos (índices desde 0): entrenamiento, desarrollo, prueba
DIVISION = {"spa": (range(0, 10), range(10, 12), range(12, 14)),
            "ita": (range(0, 4), range(4, 5), range(5, 6))}

CAPS, TRAIN, DEV, TEST = {}, {}, {}, {}
for idioma, ruta in CORPORA.items():
    CAPS[idioma] = capitulos(cargar_datos(ruta))
    tr, dv, te = DIVISION[idioma]
    TRAIN[idioma], DEV[idioma], TEST[idioma] = ([o for i in r for o in oraciones(CAPS[idioma][i])] for r in (tr, dv, te))
    palabras = sum(len(p.split()) for cap in CAPS[idioma] for p in cap)
    print(idioma, "capítulos:", len(CAPS[idioma]), "| palabras (aprox.):", palabras)

tabla(["idioma", "partición", "frases", "tokens"],
      [[i, nombre, len(d[i]), sum(map(len, d[i]))] for i in CORPORA for nombre, d in (("train", TRAIN), ("dev", DEV), ("test", TEST))])
assert all(sum(map(len, TRAIN[i])) > 20000 for i in CORPORA)

### Tres cifras que hay que mirar siempre

El tamaño de cada partición y, en la sección siguiente, la **tasa de OOV** de la prueba. Con solo unas 21 000 palabras de entrenamiento por idioma, muchas palabras de la prueba serán desconocidas: es un corpus pequeño (*ejemplo ilustrativo*) y los modelos de verdad se entrenan con muchísimo más texto.

---
## 3. El modelo completo y el problema de los ceros

Escribimos el modelo (conteos con `Counter`, probabilidades con funciones): estimación por máxima verosimilitud (`p_mle`), suavizado de Laplace (`p_laplace`), interpolación lineal de trigrama, bigrama y unigrama (`p_interp`) y un *backoff* simple (`p_backoff`) que usaremos solo para ver que no es una distribución de probabilidad.

In [ ]:
class ModeloLM:
    """Modelo de lenguaje n-grama en Python puro (Counter y dict)."""
    def __init__(self, oras_train, n=3, minimo=2):
        self.n = n
        self.vocab0 = vocabulario(oras_train, minimo)          # palabras conocidas
        oras = aplicar_vocab(oras_train, self.vocab0)
        self.V = sorted(set(t for o in oras for t in o) | {FIN, UNK})   # lo que se puede predecir (sin <s>)
        self.c, self.ctx = {}, {}
        for k in range(1, n + 1):                               # un juego de conteos por orden
            self.c[k], self.ctx[k] = contar(oras, k)

    def mapear(self, oras):
        """Aplica al texto de prueba el MISMO vocabulario del entrenamiento."""
        return aplicar_vocab(oras, self.vocab0)

    def _h(self, h, k):
        return tuple(h[len(h) - (k - 1):]) if k > 1 else ()

    def p_mle(self, w, h, k):
        """Máxima verosimilitud: c(h, w) / c(h). Vale 0 si el n-grama no se ha visto."""
        hh = self._h(h, k)
        d = self.ctx[k][hh]
        return self.c[k][hh + (w,)] / d if d else 0.0

    def p_laplace(self, w, h, k, a=1.0):
        """Suavizado de Laplace (add-a): (c(h, w) + a) / (c(h) + a·|V|)."""
        hh = self._h(h, k)
        return (self.c[k][hh + (w,)] + a) / (self.ctx[k][hh] + a * len(self.V))

    def p_interp(self, w, h, lam):
        """Interpolación lineal; lam = (peso del orden n, ..., peso del unigrama). Un orden cuyo contexto no existe cede su peso a los demás."""
        num = den = 0.0
        for peso, k in zip(lam, range(self.n, 0, -1)):
            hh = self._h(h, k)
            if self.ctx[k][hh] > 0 and peso > 0:
                num += peso * self.c[k][hh + (w,)] / self.ctx[k][hh]
                den += peso
        return num / den

    def p_backoff(self, w, h, k, alfa=0.4):
        """Stupid backoff: NO es una distribución de probabilidad (no suma 1)."""
        hh = self._h(h, k)
        if self.c[k][hh + (w,)] > 0:
            return self.c[k][hh + (w,)] / self.ctx[k][hh]
        return alfa * self.p_backoff(w, h, k - 1, alfa) if k > 1 else 0.0

def perplejidad(modelo, oras, prob):
    """exp(-media de log P) sobre los tokens de `oras` y sus </s>; no cuenta <s>. prob(w, historia) -> probabilidad."""
    logsum, N = 0.0, 0
    for o in modelo.mapear(oras):
        seq = [INI] * (modelo.n - 1) + o + [FIN]
        for i in range(modelo.n - 1, len(seq)):
            p = prob(seq[i], seq[:i])
            if p <= 0:
                return float("inf")                  # un solo cero hace infinita la perplejidad
            logsum += math.log(p)
            N += 1
    return math.exp(-logsum / N)

def generar(modelo, lam, rng, max_tokens=30):
    """Muestrea palabra a palabra desde el modelo interpolado hasta </s> o max_tokens."""
    hist = [INI] * (modelo.n - 1)
    salida = []
    while len(salida) < max_tokens:
        pesos = [modelo.p_interp(w, hist, lam) for w in modelo.V]
        w = rng.choices(modelo.V, weights=pesos, k=1)[0]
        if w == FIN:
            break
        salida.append(w)
        hist.append(w)
    return salida

In [ ]:
MODELOS = {i: ModeloLM(TRAIN[i], n=3, minimo=2) for i in CORPORA}
for i, m in MODELOS.items():
    toks_test = [t for o in TEST[i] for t in o]
    oov = sum(t not in m.vocab0 for t in toks_test) / len(toks_test)
    # n-gramas de la prueba ya vistos en el entrenamiento (con el vocabulario del entrenamiento)
    vistos = {}
    for k in (2, 3):
        cnt = [tuple(([INI] * (k - 1) + o + [FIN])[j - k + 1:j + 1]) for o in m.mapear(TEST[i]) for j in range(k - 1, len(o) + k)]
        vistos[k] = sum(x in m.c[k] for x in cnt) / len(cnt)
    print(i, "|V| =", len(m.V), "| OOV en prueba:", f"{100 * oov:.1f}%",
          "| bigramas vistos:", f"{100 * vistos[2]:.1f}%", "| trigramas vistos:", f"{100 * vistos[3]:.1f}%")
    print("   perplejidad de prueba con MLE:",
          {k: perplejidad(m, TEST[i], lambda w, h, k=k: m.p_mle(w, h, k)) for k in (1, 2, 3)})

### ¿Qué observas?

Con el máximo de verosimilitud, la perplejidad del bigrama y del trigrama es **infinita**: basta **un** bigrama de la prueba que no se vio en el entrenamiento para que una probabilidad sea 0 y todo el producto se anule. Un porcentaje alto de los trigramas de la prueba nunca aparece en el entrenamiento: contar no basta, hay que **repartir algo de probabilidad a lo no visto**. Eso es el **suavizado**. Solo el unigrama (no depende de la historia) sale finito.

---
## 4. Comprobación: las probabilidades condicionadas suman 1

Un modelo bien definido cumple $\sum_{w\in V} P(w\mid h)=1$ para **cada** historia $h$. Lo comprobamos para tres historias: la de inicio de frase, la más frecuente en el entrenamiento y una que nunca se ha visto. (`V` incluye `</s>` y `<unk>` y **no** incluye `<s>`, que nunca se predice.)

In [ ]:
def suma(m, prob, h):
    return sum(prob(w, h) for w in m.V)

LAM0 = (0.3, 0.4, 0.3)                                  # pesos de ejemplo: trigrama, bigrama, unigrama
for i, m in MODELOS.items():
    frecuente = max((k for k in m.ctx[2] if UNK not in k and INI not in k), key=m.ctx[2].get)
    historias = {"inicio": [INI, INI], "más frecuente": [INI] + list(frecuente), "nunca vista": ["zzz", "qqq"]}
    filas = []
    for nombre, h in historias.items():
        filas.append([nombre, round(suma(m, lambda w, h_: m.p_mle(w, h_, 3), h), 6),
                      round(suma(m, lambda w, h_: m.p_laplace(w, h_, 3), h), 6),
                      round(suma(m, lambda w, h_: m.p_interp(w, h_, LAM0), h), 6),
                      round(suma(m, lambda w, h_: m.p_backoff(w, h_, 3), h), 3)])
        assert abs(filas[-1][2] - 1) < 1e-9 and abs(filas[-1][3] - 1) < 1e-9      # Laplace e interpolación suman 1
        assert filas[-1][1] in (0.0, 1.0) or abs(filas[-1][1] - 1) < 1e-9
    print(i, "historia más frecuente:", frecuente)
    tabla(["historia", "MLE", "Laplace", "interpolación", "backoff"], filas, 16)

### ¿Qué observas?

- **Laplace** y la **interpolación** suman 1 en las tres historias (incluida la que nunca se vio: la interpolación cede el peso del trigrama y del bigrama al unigrama, que siempre existe).
- El **MLE** suma 1 solo si la historia se ha visto; con una historia desconocida suma 0: no está definido.
- El *backoff* simple (`alfa = 0,4`) **no suma 1** (≈ 1,1 y 1,2-1,3 en las historias vistas, ≈ 0,2 en la desconocida): da puntuaciones útiles para ordenar candidatos, pero **no es una distribución**; por eso no calculamos su perplejidad. Con descuentos y normalización (Katz, Kneser-Ney) sí lo sería; aquí solo nos quedamos con la idea de «recurrir a un n-grama más corto».

---
## 5. Perplejidad: la definición y un cálculo a mano

La **perplejidad** de un texto de $N$ tokens es $\mathrm{PP}=\exp\!\big(-\tfrac1N\sum_i \ln P(w_i\mid h_i)\big)$: la inversa de la probabilidad media por token (media geométrica). Se puede leer como «entre cuántas palabras equiprobables duda el modelo en cada paso». Aquí $N$ cuenta los tokens y los `</s>`, no los `<s>`.

Con el corpus minúsculo y un bigrama MLE, la frase de prueba `un perro duerme` tiene $P=1\cdot\frac12\cdot\frac12\cdot1=\frac14$ con $N=4$ (`un`, `perro`, `duerme`, `</s>`): $\mathrm{PP}=(1/4)^{-1/4}=\sqrt2\approx1{,}414$. Un modelo uniforme sobre $|V|$ palabras tendría $\mathrm{PP}=|V|$: sirve de referencia.

In [ ]:
chico = ModeloLM(CHICO["spa"], n=2, minimo=1)
prueba = [["un", "perro", "duerme"]]                     # dato-es
pp = perplejidad(chico, prueba, lambda w, h: chico.p_mle(w, h, 2))
pp_uniforme = perplejidad(chico, prueba, lambda w, h: 1 / len(chico.V))
print("perplejidad a mano:", math.sqrt(2), "| código:", pp)
print("|V| =", len(chico.V), "| perplejidad del modelo uniforme:", pp_uniforme)
assert abs(pp - math.sqrt(2)) < 1e-12 and abs(pp_uniforme - len(chico.V)) < 1e-9

# Laplace sobre el mismo corpus: P(pescado | un) = 1 / (4 + |V|) y la fila suma 1
p_l = chico.p_laplace("pescado", [INI, "un"], 2)
print("P_Laplace(pescado | un) = 1/", 4 + len(chico.V), "=", p_l)
assert abs(p_l - 1 / (4 + len(chico.V))) < 1e-12
assert abs(sum(chico.p_laplace(w, [INI, "un"], 2) for w in chico.V) - 1) < 1e-12

### ¿Qué observas?

La perplejidad calculada a mano y la del código coinciden (√2). Con `|V| = 9` (las 8 palabras vistas, incluida `</s>`, más `<unk>`), el modelo uniforme tendría perplejidad 9. El suavizado de Laplace (`+1` en cada n-grama) convierte `P(pescado | un) = 0` en `1/13`, pero se lo **quita** a los n-gramas vistos: `P(gato | un)` baja de 1/2 a 3/13.

---
## 6. Experimento guiado 1 — orden del modelo, suavizado y perplejidad de prueba

Comparamos, sobre el mismo vocabulario y la misma tokenización, unigrama, bigrama y trigrama con Laplace (`a = 1`), con Laplace `a = 0,01` y con interpolación. Los pesos de la interpolación **no** se eligen mirando la prueba: se buscan en una rejilla sobre el conjunto de **desarrollo**.

In [ ]:
# Rejilla de pesos (trigrama, bigrama, unigrama) que suman 1, con peso de unigrama ≥ 0,1
REJILLA = [(a / 10, b / 10, (10 - a - b) / 10) for a in range(0, 9) for b in range(0, 10 - a) if 10 - a - b >= 1]

MEJOR = {}
for i, m in MODELOS.items():
    puntos = [(perplejidad(m, DEV[i], lambda w, h, lam=lam: m.p_interp(w, h, lam)), lam) for lam in REJILLA]
    MEJOR[i] = min(puntos)[1]
    print(i, "mejores pesos en desarrollo:", MEJOR[i], "| PP dev =", round(min(puntos)[0], 1))

filas = []
for i, m in MODELOS.items():
    fila = [i]
    for k in (1, 2, 3):
        fila.append(round(perplejidad(m, TEST[i], lambda w, h, k=k: m.p_laplace(w, h, k, 1.0)), 1))
    fila.append(round(perplejidad(m, TEST[i], lambda w, h: m.p_laplace(w, h, 3, 0.01)), 1))
    fila.append(round(perplejidad(m, TEST[i], lambda w, h: m.p_interp(w, h, MEJOR[i])), 1))
    fila.append(round(perplejidad(m, TRAIN[i], lambda w, h: m.p_mle(w, h, 3)), 1))
    filas.append(fila)
tabla(["idioma", "1-g Laplace", "2-g Laplace", "3-g Laplace", "3-g a=0,01", "interpolado", "3-g MLE train"], filas, 15)
assert all(f[5] < min(f[1], f[2], f[3]) for f in filas)          # la interpolación gana en prueba

### ¿Qué observas?

Con estos datos (*ejemplo ilustrativo*, un corpus pequeño):

- Con Laplace `a = 1` el trigrama es **peor** que el bigrama y este peor que el unigrama: con |V| ≈ 1 600 y pocos datos, «+1» reparte demasiada probabilidad a lo no visto (el suavizado de Laplace es demasiado agresivo para n-gramas de orden alto). Con `a = 0,01` mejora, pero sigue detrás.
- La **interpolación** (pesos elegidos en desarrollo, con mucho peso para los órdenes bajos) gana en la prueba en los dos idiomas.
- La perplejidad del trigrama MLE **en el entrenamiento** es muy baja (el modelo memoriza), mientras que en la prueba era infinita: medir siempre sobre datos apartados.
- No mires las cifras de español e italiano como una carrera: en la sección siguiente vemos por qué no se pueden comparar.

---
## 7. Experimento guiado 2 — la perplejidad solo es comparable con el mismo vocabulario

Sobre **el mismo texto de prueba y el mismo modelo**, cambiamos solo el umbral de OOV (una palabra entra en el vocabulario si aparece al menos `minimo` veces) y, aparte, la tokenización (con signos de puntuación como tokens). Los pesos de interpolación son los de desarrollo. Antes de ejecutar, predice: ¿debería cambiar la perplejidad?

In [ ]:
UMBRALES = [1, 2, 3, 5, 10, 20, 50, 10 ** 6]
RES_UMBRAL = {}
for i in CORPORA:
    filas = []
    for minimo in UMBRALES:
        m = ModeloLM(TRAIN[i], n=3, minimo=minimo)
        toks = [t for o in TEST[i] for t in o]
        oov = 100 * sum(t not in m.vocab0 for t in toks) / len(toks)
        pp = perplejidad(m, TEST[i], lambda w, h: m.p_interp(w, h, MEJOR[i]))
        filas.append([minimo if minimo < 10 ** 6 else "todo <unk>", len(m.V), round(oov, 1), round(pp, 1)])
    RES_UMBRAL[i] = filas
    print(i, "(mismo texto de prueba, mismos pesos; solo cambia el vocabulario)")
    tabla(["minimo", "|V|", "OOV prueba %", "perplejidad"], filas, 16)

# Tokenización con puntuación: otro «vocabulario» sobre el mismo texto
for i in CORPORA:
    tr, dv, te = DIVISION[i]
    T = [o for k in tr for o in oraciones(CAPS[i][k], puntuacion=True)]
    P = [o for k in te for o in oraciones(CAPS[i][k], puntuacion=True)]
    m = ModeloLM(T, n=3, minimo=2)
    print(i, "con puntuación como tokens: |V| =", len(m.V),
          "| perplejidad =", round(perplejidad(m, P, lambda w, h: m.p_interp(w, h, MEJOR[i])), 1))

fig, ax = plt.subplots()
for i, filas in RES_UMBRAL.items():
    ax.plot([f[2] for f in filas[1:-1]], [f[3] for f in filas[1:-1]], marker="o", label=i)
ax.set_xlabel("% de OOV en la prueba"); ax.set_ylabel("perplejidad"); ax.set_yscale("log")
ax.set_title("Más OOV, «mejor» perplejidad: una ilusión"); ax.legend()
plt.show()
assert RES_UMBRAL["spa"][-1][3] < 2                       # todo <unk>: perplejidad casi 1
assert RES_UMBRAL["spa"][0][3] == float("inf")            # minimo=1: <unk> nunca visto en entrenamiento

### ¿Qué observas?

- Con el **mismo texto de prueba**, subir el umbral de OOV hace **bajar la perplejidad** (en español de 78,0 a 14,1 con `minimo = 20`; en italiano de 102,6 a 17,4) y hasta ~1 si todo se convierte en `<unk>`. El modelo no ha mejorado: ha **cambiado la pregunta** (cada vez hay que predecir menos palabras distintas, y `<unk>` es muy fácil de predecir). Un modelo que lo convierte todo en `<unk>` tendría perplejidad casi 1 y no sabría nada del idioma.
- Con `minimo = 1` no existe ningún `<unk>` en el entrenamiento y cualquier palabra nueva de la prueba tiene probabilidad 0: perplejidad infinita. Tratar el OOV es **obligatorio**, y hay que declararlo.
- Añadir la puntuación como tokens también baja la perplejidad (los signos son fáciles de predecir y cuentan como tokens): otra vez, otro vocabulario.
- **Conclusión:** comparar perplejidades solo tiene sentido con el mismo vocabulario, la misma tokenización y el mismo tratamiento del OOV (y, mejor aún, el mismo texto de prueba). **Comparar el español con el italiano no tiene sentido**: cambian el texto, el vocabulario (|V|), la tasa de OOV, la morfología, la tokenización (los apóstrofos italianos) y el autor y el género. Las cifras de las dos lenguas de esta clase se leen por separado.

---
## 8. Experimento guiado 3 — capítulos contiguos frente a frases al azar

Repetimos la evaluación con un reparto **al azar frase por frase** (5 semillas), con los mismos tamaños de entrenamiento y prueba y las mismas reglas de OOV, y comparamos con el reparto por capítulos. Miramos tres indicadores: la tasa de OOV en la prueba, la cobertura de bigramas y trigramas de la prueba, y la perplejidad.

In [ ]:
def indicadores(train, test, lam):
    m = ModeloLM(train, n=3, minimo=2)
    toks = [t for o in test for t in o]
    oov = 100 * sum(t not in m.vocab0 for t in toks) / len(toks)
    cob = []
    for k in (2, 3):
        cnt = [tuple(([INI] * (k - 1) + o + [FIN])[j - k + 1:j + 1]) for o in m.mapear(test) for j in range(k - 1, len(o) + k)]
        cob.append(100 * sum(x in m.c[k] for x in cnt) / len(cnt))
    return [round(oov, 1), round(cob[0], 1), round(cob[1], 1), round(perplejidad(m, test, lambda w, h: m.p_interp(w, h, lam)), 1)]

def reparto_azar(todas, n_train, n_test, semilla):
    """Mezcla las frases y reparte hasta alcanzar los tamaños (en tokens) de la división por capítulos."""
    rng = random.Random(semilla)
    o = todas[:]
    rng.shuffle(o)
    prueba, resto, c = [], [], 0
    for s in o:
        if c < n_test:
            prueba.append(s); c += len(s)
        else:
            resto.append(s)
    train, c = [], 0
    for s in resto:
        if c < n_train:
            train.append(s); c += len(s)
    return train, prueba

for i in CORPORA:
    todas = [o for cap in CAPS[i] for o in oraciones(cap)]
    n_train, n_test = sum(map(len, TRAIN[i])), sum(map(len, TEST[i]))
    filas = [["capítulos", ""] + indicadores(TRAIN[i], TEST[i], MEJOR[i])]
    for semilla in range(5):
        tr, te = reparto_azar(todas, n_train, n_test, semilla)
        filas.append(["al azar", semilla] + indicadores(tr, te, MEJOR[i]))
    print(i)
    tabla(["reparto", "semilla", "OOV %", "bigr. vistos %", "trigr. vistos %", "perplejidad"], filas, 16)

### ¿Qué observas?

- En **español**, el reparto al azar reduce la tasa de OOV de la prueba (de 22,7 % a 16,6-18,4 %): la prueba comparte con el entrenamiento personajes, lugares y pasajes de los mismos capítulos. Es una **filtración** (*leakage*) de información del futuro hacia el entrenamiento.
- Pero las demás cifras **no dan una señal limpia**: la cobertura de bigramas y trigramas no sube y la perplejidad sube (83-93 frente a 78,0). Con menos OOV hay menos `<unk>` (baratos de predecir), y además la prueba por capítulos y la prueba al azar son textos distintos. La variación entre semillas (83 a 93) es del tamaño de la diferencia con el reparto por capítulos: **dos perplejidades con distinta tasa de OOV o distinto texto de prueba no son comparables** (mismo aviso de la sección 7).
- En **italiano**, en este extracto, casi no hay diferencias. Que no se vean no prueba que no exista filtración: un extracto pequeño y homogéneo no la detecta.
- **Regla práctica:** el reparto no se elige por la cifra que sale mejor, sino por lo que imita el uso real: el modelo se aplicará a texto que llega después, así que se divide en bloques contiguos (capítulos, meses…) y se declara. Un corpus literario antiguo tampoco es representativo del español o el italiano de hoy: las cifras valen para este libro.

---
## 9. Lengua de hace dos siglos frente a lengua de hoy

Probamos cada modelo sobre reseñas **modernas** del curso (`texto_limpio` de `resenas_es.csv` y `resenas_it.csv`, sintéticas): el mismo vocabulario y la misma tokenización, otro dominio y otra época.

In [ ]:
import csv
for i, archivo in (("spa", "resenas_es.csv"), ("ita", "resenas_it.csv")):
    with open(cargar_datos(archivo), encoding="utf-8", newline="") as f:
        resenas = [fila["texto_limpio"] for fila in csv.DictReader(f)]
    modernas = oraciones(resenas)
    m = MODELOS[i]
    toks = [t for o in modernas for t in o]
    oov_mod = 100 * sum(t not in m.vocab0 for t in toks) / len(toks)
    toks_te = [t for o in TEST[i] for t in o]
    oov_te = 100 * sum(t not in m.vocab0 for t in toks_te) / len(toks_te)
    print(i, "OOV en reseñas modernas:", f"{oov_mod:.1f}%", "| OOV en la prueba literaria:", f"{oov_te:.1f}%")
    print("   palabras de hoy desconocidas para el modelo (las 8 más frecuentes):", [w for w, _ in Counter(t for t in toks if t not in m.vocab0).most_common(8)])
    assert oov_mod > oov_te

### ¿Qué observas?

La tasa de OOV sobre texto moderno es mayor (≈ 38 % frente a ≈ 22,7 % en español; ≈ 40 % frente a ≈ 20,9 % en italiano) que sobre la prueba literaria de la misma obra: el modelo entrenado con un libro clásico conoce poco del vocabulario de hoy (mira las palabras desconocidas más frecuentes que se imprimen). No lo usaríamos para nada actual. Y esta cifra no es una perplejidad comparable con las anteriores.

---
## 10. Generar texto por muestreo y sus límites

Para generar, partimos de `<s> <s>` y en cada paso **muestreamos** la palabra siguiente según $P(w\mid h)$ hasta obtener `</s>`. Con la semilla fija el resultado es reproducible. Comparamos tres modelos (unigrama, bigrama y trigrama, por los pesos de la interpolación) y medimos cuánto del texto generado son trigramas copiados del entrenamiento.

In [ ]:
ORDENES = {"unigrama": (0.0, 0.0, 1.0), "bigrama": (0.0, 0.9, 0.1), "trigrama": (0.8, 0.15, 0.05)}
for i, m in MODELOS.items():
    c3 = contar(m.mapear(TRAIN[i]), 3)[0]
    filas = []
    print(i)
    for nombre, lam in ORDENES.items():
        rng = random.Random(2026)                          # semilla fija
        muestras = [generar(m, lam, rng, 30) for _ in range(30)]
        tri = [tuple(([INI, INI] + g)[j - 2:j + 1]) for g in muestras for j in range(2, len(g) + 2)]
        copiados = 100 * sum(x in c3 for x in tri) / len(tri)
        unk = 100 * sum(g.count(UNK) for g in muestras) / sum(map(len, muestras))
        filas.append([nombre, round(copiados, 1), round(unk, 1), round(sum(map(len, muestras)) / len(muestras), 1)])
        print("  ", nombre, ":", " ".join(generar(m, lam, random.Random(7), 25)))
    tabla(["modelo", "trigr. en train %", "<unk> %", "long. media"], filas, 22)
    assert filas[2][1] > filas[1][1] > filas[0][1]

### ¿Qué observas?

- El **unigrama** produce una «ensalada de palabras»; el **bigrama** produce pares plausibles pero sin sentido global; el **trigrama** suena más fluido porque **copia** fragmentos del entrenamiento (más de un 80 % de sus trigramas ya estaban en el texto): con tan poco texto, un trigrama casi memoriza.
- Aparecen `<unk>` en el texto: el modelo no sabe qué palabra hay detrás.
- Nada garantiza **coherencia a larga distancia** (concordancia, tema, argumento): el contexto es de solo una o dos palabras. Además, con la semilla fija todo es reproducible, pero **otra semilla da otro texto**: no hay un texto «correcto».
- Los modelos de lenguaje modernos (redes neuronales, LLM) siguen la misma idea (predecir la palabra siguiente y muestrear), pero con contextos mucho más largos.

---
## Resumen

| Concepto | Idea clave |
|---|---|
| Regla de la cadena y Markov | la probabilidad de la frase es un producto de condicionadas, recortadas a n−1 palabras previas |
| Conteo y tokens especiales | `<s>`, `</s>` y `<unk>` se declaran; P = c(h, w) / c(h) |
| Suavizado | el MLE da 0 a lo no visto; Laplace es agresivo; la interpolación combina órdenes; el *backoff* simple no es una distribución |
| Comprobación | las condicionadas suman 1 para cada historia |
| Perplejidad | solo comparable con el mismo vocabulario, tokenización y tratamiento del OOV; ES frente a IT no se comparan |
| División | por bloques contiguos, no al azar; corpus antiguo ≠ lengua de hoy |
| Generación | muestrear reproduce localmente el corpus, sin coherencia global |

**Siguiente:** en la clase 9 pasamos de las secuencias a los **temas** de un corpus (*topic modeling*: LSA y LDA). Después, la práctica de esta clase.